# RM Simulator (SOCat)

Simulate a **Rossiter–McLaughlin** radial-velocity sequence with the Hirano et al. (2011) model. Runs entirely in your browser (Pyodide) — the transit model is a pure-numpy quadratic limb-darkening occultation, no `pytransit`/`numba`/`ironman` needed.

Edit the parameters below and re-run, or use the interactive sliders at the bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from socat import rm_model   # vendored pure-numpy Hirano (2011) RM + transit

def t0_to_tp(t0, period, ecc, omega):
    """Mid-transit time -> time of periastron (inverse of rm_model._tp_to_t0)."""
    f = 0.5*np.pi - omega
    E = 2*np.arctan(np.tan(0.5*f)*np.sqrt((1-ecc)/(1+ecc)))
    M = E - ecc*np.sin(E)
    return t0 - period*M/(2*np.pi)

def rm_curve(bjd, t0, period, ar, inc_deg, p, e, omega_deg, lam_deg,
             vsini_kms, u1, u2):
    """RM anomaly [m/s] vs bjd. Angles in deg, vsini in km/s."""
    omega = np.radians(omega_deg)
    tp = t0_to_tp(t0, period, e, omega)
    return rm_model.hirano2011_rm(
        bjd, tp, period, e, omega, np.radians(inc_deg), ar, p,
        u1, u2, vsini_kms*1e3, np.radians(lam_deg))

print('rm_model loaded — no pytransit:', 'pytransit' not in dir(rm_model))

## System & observation parameters

In [ ]:
# --- planet / star ---
period   = 18.09537     # orbital period [days]
t0       = 2458529.32785 # mid-transit time [BJD]
aRs      = 19.01        # a / R*
inc      = 84.25        # inclination [deg]
p        = 0.069        # Rp / R*
e        = 0.72         # eccentricity
omega    = 60.5         # argument of periastron [deg]
lam      = 1.2          # projected obliquity [deg]
vsini    = 20.2         # v sin i* [km/s]
u1, u2   = 0.327, 0.331 # quadratic limb darkening

# --- observation ---
rv_prec  = 10.0         # per-point RV precision [m/s]
exp_time = 300.0        # cadence [s]
win_hr   = 4.0          # half-window around transit [hours]

In [ ]:
# observed (noisy) samples
cad = exp_time/86400.0
bjd_obs = np.arange(t0 - win_hr/24.0, t0 + win_hr/24.0, cad)
model_obs = rm_curve(bjd_obs, t0, period, aRs, inc, p, e, omega, lam, vsini, u1, u2)
rng = np.random.default_rng(0)
rv_obs = model_obs + rng.normal(0.0, rv_prec, size=model_obs.size)
err_obs = np.full_like(rv_obs, rv_prec)

# smooth model
bjd_hr_grid = np.linspace(-win_hr, win_hr, 600)
bjd_grid = t0 + bjd_hr_grid/24.0
model_hr = rm_curve(bjd_grid, t0, period, aRs, inc, p, e, omega, lam, vsini, u1, u2)

hr_obs = (bjd_obs - t0)*24.0
fig, ax = plt.subplots(figsize=(7,4.2))
ax.plot(bjd_hr_grid, model_hr, color='firebrick', ls='--', label='Model')
ax.errorbar(hr_obs, rv_obs, yerr=err_obs, fmt='.', color='silver', zorder=1)
ax.scatter(hr_obs, rv_obs, color='lightblue', edgecolor='k', s=22, zorder=2, label='Simulated RV')
ax.axhline(0, color='0.7', lw=0.8)
ax.set_xlabel('Hours from mid-transit'); ax.set_ylabel('RV [m/s]')
ax.legend(); ax.set_title('Rossiter–McLaughlin — Hirano (2011)')
plt.tight_layout(); plt.show()
print(f'RM semi-amplitude ~ {np.ptp(model_hr)/2:.1f} m/s')

## Interactive (optional)
Needs `ipywidgets`, which isn't pre-installed in this kernel — the cell below installs it (one-time, ~a few seconds). The static plot above already works without it; drag the sliders (especially **λ**) to see how the RM shape changes.

In [ ]:
%pip install -q ipywidgets

In [ ]:
from ipywidgets import interact, FloatSlider

def _plot(lam=1.2, vsini=20.2, b=0.0, p=0.069):
    inc_ = np.degrees(np.arccos(b/aRs))
    g = np.linspace(-win_hr, win_hr, 500)
    m = rm_curve(t0 + g/24.0, t0, period, aRs, inc_, p, e, omega, lam, vsini, u1, u2)
    plt.figure(figsize=(7,4))
    plt.plot(g, m, color='firebrick'); plt.axhline(0, color='0.7', lw=0.8)
    plt.xlabel('Hours from mid-transit'); plt.ylabel('RV [m/s]')
    plt.title(f'λ={lam:.0f}°, vsini={vsini:.1f} km/s, b={b:.2f}, Rp/R*={p:.3f}')
    plt.tight_layout(); plt.show()

interact(_plot,
         lam=FloatSlider(value=1.2, min=-180, max=180, step=5, description='λ [deg]'),
         vsini=FloatSlider(value=20.2, min=1, max=80, step=1, description='vsini'),
         b=FloatSlider(value=0.0, min=0.0, max=0.95, step=0.05, description='impact b'),
         p=FloatSlider(value=0.069, min=0.02, max=0.2, step=0.005, description='Rp/R*'));